# AI CCTV Investigator — IRRA 4-Stripe Cross-Modal Trainer (Stages 2 & 3)

This self-contained notebook trains and evaluates the **CLIP ViT-B/16 + IRRA 4-Stripe Horizontal Attention Head** pipeline on Kaggle P100 or Google Colab T4 GPUs:
- **Stage 2 (`backbone_clip.py`)**: Dual `open_clip` `ViT-B-16` (`openai` weights, `EMBED_DIM = 512`, strictly L2-normalized).
- **Stage 3 (`stripe_attention_head.py`, `losses.py`)**: 4-stripe anatomical partitioning (Head, Torso, Legs, Accessories/Bag) trained with `InfoNCE + SDM Loss` (`tau = 0.02`, `lambda_sdm = 1.0`).
- **Parameter-Efficient Fine-Tuning (`train.py`)**: Early ViT blocks stay frozen so fine-tuning finishes in **3–4 hours** on a single P100/T4.
- **Evaluation (`evaluate.py`)**: Logs **Zero-Shot CLIP** vs. **IRRA 4-Stripe** on `Recall@1`, `Recall@5`, `Recall@10`, and `mAP`.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/SoldierMeer/ai-cctv-investigator.git"
BRANCH = "feat/dl-core-clip-irra"

# Locate existing monorepo or clone in Colab/Kaggle runtime
candidates = [
    Path.cwd(),
    Path.cwd().parents[2] if len(Path.cwd().parents) >= 3 else Path.cwd(),
    Path("/kaggle/working/ai-cctv-investigator"),
    Path("/content/ai-cctv-investigator"),
]
repo_root = next((p for p in candidates if (p / "apps" / "dl-core" / "src").exists()), None)

if repo_root is None:
    target_dir = Path("/kaggle/working/ai-cctv-investigator") if Path("/kaggle/working").exists() else Path("/content/ai-cctv-investigator")
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(target_dir)], check=True)
    repo_root = target_dir

os.chdir(repo_root)
print(f"Repository root: {repo_root}")

# Install dl-core dependencies if open_clip is missing
try:
    import open_clip  # noqa: F401
    import einops  # noqa: F401
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "apps/dl-core/requirements.txt"])

for p in [str(repo_root / "apps" / "dl-core" / "src"), str(repo_root / "packages" / "shared-contracts" / "python")]:
    if p not in sys.path:
        sys.path.insert(0, p)

import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Active device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "CPU")

In [ ]:
from torch.utils.data import DataLoader
from datasets import PersonSearchDataset
from evaluate import RetrievalEvaluator
from backbone_clip import CLIPBackbone
from stripe_attention_head import IRRAStripeHead
from train import TrainConfig, evaluate_retrieval, run_training
from embed_extractor import EmbedExtractor

# Toggle `USE_MOCK_DATA = False` when real CUHK-PEDES / ICFG-PEDES / RSTPReid / campus_custom files are mounted under `data/`
DATASET_NAME = "cuhk_pedes"  # options: 'cuhk_pedes', 'icfg_pedes', 'rstpreid', 'campus_custom'
USE_MOCK_DATA = True
PRETRAINED_WEIGHTS = "openai"  # set to None for offline random-init smoke check

val_dataset = PersonSearchDataset(
    dataset_name=DATASET_NAME,
    split="test",
    mock=USE_MOCK_DATA,
    mock_size=32,
    mock_num_pids=8,
)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0)
print(f"Loaded {DATASET_NAME} validation split: {len(val_dataset)} samples (mock={USE_MOCK_DATA})")

In [ ]:
# Baseline 1: Zero-Shot CLIP ViT-B/16 Evaluation
evaluator = RetrievalEvaluator()
zeroshot_backbone = CLIPBackbone(model_name="ViT-B-16", pretrained=PRETRAINED_WEIGHTS, device=device)
zeroshot_metrics = evaluate_retrieval(
    backbone=zeroshot_backbone,
    stripe_head=None,
    val_loader=val_loader,
    evaluator=evaluator,
    use_stripe_head=False,
)
print("Zero-Shot CLIP Metrics:")
for k, v in zeroshot_metrics.items():
    print(f"  {k}: {v * 100:.2f}%")

In [ ]:
# Stage 3: Parameter-Efficient IRRA 4-Stripe Fine-Tuning (AdamW + Cosine Scheduler)
ckpt_path = repo_root / "apps" / "dl-core" / "checkpoints" / "irra_stripe_best.pt"

cfg = TrainConfig(
    dataset_name=DATASET_NAME,
    train_split="train",
    val_split="test",
    epochs=3 if USE_MOCK_DATA else 30,
    batch_size=16 if USE_MOCK_DATA else 64,
    lr_head=1e-4,
    lr_backbone=1e-5,
    tau=0.02,
    lambda_sdm=1.0,
    trainable_visual_blocks=2,
    trainable_text_blocks=2,
    pretrained=PRETRAINED_WEIGHTS,
    device=device,
    checkpoint_path=str(ckpt_path),
    mock=USE_MOCK_DATA,
    mock_train_size=32,
    mock_val_size=32,
)

train_summary = run_training(cfg)
print(f"Best checkpoint saved to: {train_summary['checkpoint_path']}")

In [ ]:
# Compare Zero-Shot CLIP vs. Fine-Tuned IRRA 4-Stripe using saved checkpoint
extractor = EmbedExtractor(
    checkpoint_path=ckpt_path,
    pretrained=PRETRAINED_WEIGHTS,
    device=device,
)

irra_metrics = evaluate_retrieval(
    backbone=extractor.backbone,
    stripe_head=extractor.stripe_head,
    val_loader=val_loader,
    evaluator=evaluator,
    use_stripe_head=True,
)

print(f"{'Metric':<12} | {'Zero-Shot CLIP':>15} | {'IRRA 4-Stripe':>15} | {'Delta':>10}")
print("-" * 60)
for metric in ["Recall@1", "Recall@5", "Recall@10", "mAP"]:
    zs_val = train_summary["zero_shot_metrics"][metric] * 100.0
    irra_val = irra_metrics[metric] * 100.0
    print(f"{metric:<12} | {zs_val:>14.2f}% | {irra_val:>14.2f}% | {irra_val - zs_val:>+9.2f}%")

# Sanity check EmbedExtractor (1, 512) float32 unit vector output
import numpy as np
q_vec = extractor.encode_text_query("Person wearing a red jacket and black jeans carrying a backpack")
assert q_vec.shape == (1, 512) and q_vec.dtype == np.float32
assert np.isclose(np.linalg.norm(q_vec), 1.0, atol=1e-5)
print(f"\nVerified query embedding shape={q_vec.shape}, dtype={q_vec.dtype}, norm={np.linalg.norm(q_vec):.6f}")